In [51]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder , StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression ,Lasso , Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [25]:
df=pd.read_csv("insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0.0,yes,southwest,16884.92400
1,18.0,male,33.770,1.0,no,southeast,1725.55230
2,28.0,male,33.000,3.0,no,southeast,4449.46200
3,33.0,male,22.705,0.0,no,northwest,21984.47061
4,32.0,male,28.880,0.0,no,northwest,3866.85520


In [26]:
df.shape

(1338, 7)

In [27]:
df.columns

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges'], dtype='object')

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1296 non-null   float64
 1   sex       1329 non-null   object 
 2   bmi       1296 non-null   float64
 3   children  1314 non-null   float64
 4   smoker    1305 non-null   object 
 5   region    1054 non-null   object 
 6   charges   1314 non-null   float64
dtypes: float64(4), object(3)
memory usage: 73.3+ KB


In [29]:
df.describe()

,age,bmi,children,charges
count,1296.000000,1296.000000,1314.000000,1314.000000
mean,39.225309,30.655197,1.092846,13259.233168
std,14.034162,6.117612,1.205684,12136.291497
min,18.000000,15.960000,0.000000,1121.873900
25%,26.000000,26.220000,0.000000,4724.369462
50%,39.000000,30.380000,1.000000,9289.083100
75%,51.000000,34.600000,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [30]:
for col in df.select_dtypes('object'):
    print(col,df[col].unique(),'\n',"_"*35)

sex ['female' 'male' nan] 
 ___________________________________
smoker ['yes' 'no' nan] 
 ___________________________________
region ['southwest' 'southeast' 'northwest' 'northeast' nan] 
 ___________________________________


# **Handle NULL Values**

In [31]:
df.isna().sum().sort_values(ascending=False)

region      284
age          42
bmi          42
smoker       33
children     24
charges      24
sex           9
dtype: int64

In [32]:
df.isna().sum().sum()

458

In [33]:
df.dropna(subset='charges',axis=0,inplace=True)

In [34]:
df.reset_index(inplace=True,drop=True)

In [35]:
df.shape[0]-df.isna().sum().sum(),df.shape[0]

(916, 1314)

In [36]:
X=df.drop('charges',axis=1)
y=df['charges']
X_train,X_test,y_train,y_test=train_test_split(X,y ,test_size=0.20)

In [38]:
cat_columns=X.select_dtypes('object').columns
impute=SimpleImputer(missing_values=np.nan, strategy='most_frequent')
X_train[cat_columns]=impute.fit_transform(X_train[cat_columns])
X_test[cat_columns]=impute.transform(X_test[cat_columns])

In [39]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1051 entries, 145 to 979
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1024 non-null   float64
 1   sex       1051 non-null   object 
 2   bmi       1026 non-null   float64
 3   children  1039 non-null   float64
 4   smoker    1051 non-null   object 
 5   region    1051 non-null   object 
dtypes: float64(3), object(3)
memory usage: 89.8+ KB


In [41]:
num_columns=['age','bmi']
impute=SimpleImputer(missing_values=np.nan, strategy='mean')
X_train[num_columns]=impute.fit_transform(X_train[num_columns])
X_test[num_columns]=impute.transform(X_test[num_columns])

In [44]:
impute=SimpleImputer(missing_values=np.nan, strategy='median')
X_train[['children']]=impute.fit_transform(X_train[['children']])
X_test[['children']]=impute.transform(X_test[['children']])

# **Encoding & Scaling**

In [48]:
ord=OrdinalEncoder()
X_train[cat_columns]=ord.fit_transform(X_train[cat_columns])
X_test[cat_columns]=ord.transform(X_test[cat_columns])

In [49]:
cols=X.select_dtypes('number').columns
scaler=StandardScaler()
X_train[cols]=scaler.fit_transform(X_train[cols])
X_test[cols]=scaler.transform(X_test[cols])

# **Modeling**

In [54]:
models={
    "LinearRegression":LinearRegression(),
    "Lasso":Lasso(),
    "Ridge":Ridge(),
    "RandomForestRegressor":RandomForestRegressor(n_estimators=10,criterion='squared_error',max_depth=10),
    "XGBRegressor":XGBRegressor(n_rstimators=10,learning_rate=0.1, random_state=42)     
    }

In [55]:
accuracy=[]
for model_name , model in models.items():
    model.fit(X_train,y_train)
    MAE_Train = mean_absolute_error(y_train,model.predict(X_train))
    MAE_Test  =  mean_absolute_error(y_test,model.predict(X_test))

    MSE_Train = mean_squared_error(y_train,model.predict(X_train))
    MSE_Test  =  mean_squared_error(y_test,model.predict(X_test))

    RMSE_Train = np.sqrt(MSE_Train)
    RMSE_Test  = np.sqrt(MSE_Test) 

    R_Train = r2_score(y_train,model.predict(X_train))
    R_Test  =  r2_score(y_test,model.predict(X_test))
    
    accuracy.append([MAE_Train,MSE_Train,RMSE_Train,R_Train,MAE_Test,MSE_Test,RMSE_Test,R_Test])
    


d:\Anaconda\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:13:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "n_rstimators" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [56]:
pd.DataFrame(accuracy,columns=["MAE_Train","MSE_Train","RMSE_Train","R_Train","MAE_Test","MSE_Test","RMSE_Test","R_Test"],index=models.keys())

,MAE_Train,MSE_Train,RMSE_Train,R_Train,MAE_Test,MSE_Test,RMSE_Test,R_Test
LinearRegression,4422.313759,4.143553e+07,6437.043639,0.724126,4179.835665,3.389757e+07,5822.162106,0.748603
Lasso,4422.528943,4.143554e+07,6437.044680,0.724126,4180.399702,3.390052e+07,5822.414964,0.748581
Ridge,4432.018462,4.143853e+07,6437.276869,0.724106,4192.442411,3.397827e+07,5829.087955,0.748005
RandomForestRegressor,1496.221855,8.483207e+06,2912.594505,0.943520,2945.824591,2.479596e+07,4979.554588,0.816104
XGBRegressor,1391.972973,7.086577e+06,2662.062500,0.952818,2874.848826,2.152286e+07,4639.273701,0.840379
